In [14]:
# Import libraries needed for data handling, numerical operations, and standardization
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [15]:
# Load each CSV into a pandas DataFrame so we can work with it as a table
g1 = pd.read_csv('Plant_1_Generation_Data.csv')
w1 = pd.read_csv('Plant_1_Weather_Sensor_Data.csv')
g2 = pd.read_csv('Plant_2_Generation_Data.csv')
w2 = pd.read_csv('Plant_2_Weather_Sensor_Data.csv')

print(g1.shape, w1.shape, g2.shape, w2.shape)

(68778, 7) (3182, 6) (67698, 7) (3259, 6)


In [16]:
# convert all DATE_TIME columns to the same datetime format
g1['DATE_TIME'] = pd.to_datetime(g1['DATE_TIME'], format='%d-%m-%Y %H:%M')
w1['DATE_TIME'] = pd.to_datetime(w1['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
g2['DATE_TIME'] = pd.to_datetime(g2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
w2['DATE_TIME'] = pd.to_datetime(w2['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

print(g1['DATE_TIME'].iloc[0], w1['DATE_TIME'].iloc[0])

2020-05-15 00:00:00 2020-05-15 00:00:00


In [17]:
# Weather SOURCE_KEY means a different thing than generation SOURCE_KEY (1 sensor vs 22 inverters)
# drop the duplicate copy (PLANT_ID)
w1 = w1.drop(columns=['SOURCE_KEY', 'PLANT_ID'])
w2 = w2.drop(columns=['SOURCE_KEY', 'PLANT_ID'])

In [18]:
# Merge each plant's generation data with its own weather data using DATE_TIME as the key
plant1 = pd.merge(g1, w1, on='DATE_TIME', how='left')
plant2 = pd.merge(g2, w2, on='DATE_TIME', how='left')

print(plant1.shape, plant2.shape)

(68778, 10) (67698, 10)


In [19]:
# Stack Plant 1 and Plant 2 into a single dataset
full_df = pd.concat([plant1, plant2], ignore_index=True)
print(full_df.shape)

(136476, 10)


In [20]:
# Check how many missing (NaN) values exist in each column
print(full_df.isnull().sum())

DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
AMBIENT_TEMPERATURE    4
MODULE_TEMPERATURE     4
IRRADIATION            4
dtype: int64


In [21]:
# Sort by plant, inverter, and time so interpolation uses correct time-adjacent values
full_df = full_df.sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

In [22]:
# Estimate missing weather readings using linear interpolation between neighboring timestamps
weather_cols = ['AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
full_df[weather_cols] = full_df.groupby('PLANT_ID')[weather_cols].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

print(full_df[weather_cols].isnull().sum())

AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64


In [23]:
# Standardize features to mean=0, std=1 so all features are on a comparable scale
feature_cols = ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'AC_POWER']
scaler = StandardScaler()
scaled_values = scaler.fit_transform(full_df[feature_cols])

scaled_col_names = [c + '_scaled' for c in feature_cols]
full_df[scaled_col_names] = scaled_values

full_df[feature_cols + scaled_col_names].head()

,IRRADIATION,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,AC_POWER,IRRADIATION_scaled,AMBIENT_TEMPERATURE_scaled,MODULE_TEMPERATURE_scaled,AC_POWER_scaled
0,0.0,25.184316,22.857507,0.0,-0.755031,-0.405101,-0.767865,-0.722823
1,0.0,25.084589,22.761668,0.0,-0.755031,-0.430690,-0.775984,-0.722823
2,0.0,24.935753,22.592306,0.0,-0.755031,-0.468880,-0.790332,-0.722823
3,0.0,24.846130,22.360852,0.0,-0.755031,-0.491876,-0.809941,-0.722823
4,0.0,24.621525,22.165423,0.0,-0.755031,-0.549507,-0.826497,-0.722823


In [24]:
# Save the final merged, cleaned, and scaled dataset, then download it locally
full_df.to_csv('merged_cleaned_scaled.csv', index=False)

from google.colab import files
files.download('merged_cleaned_scaled.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>